In [ ]:
from google.colab import userdata
AUTH_TOKEN_NGROK = userdata.get('NGROK_AUTH_TOKEN')

# Installation

In [ ]:
# Add PHP 8.3 PPA and install PHP + required extensions
!sudo apt-get update -qq
!sudo add-apt-repository ppa:ondrej/php -y -qq
!sudo apt-get update -qq
!sudo apt-get install -y php8.3-cli php8.3-common php8.3-mbstring php8.3-gd php8.3-intl php8.3-xml php8.3-mysql php8.3-zip php8.3-curl php8.3-bcmath
# Set PHP 8.3 as default
!sudo update-alternatives --set php /usr/bin/php8.3 2>/dev/null || true
!php -v

In [ ]:
# Install Composer
!wget -q https://getcomposer.org/download/latest-stable/composer.phar
!sudo mv composer.phar /usr/local/bin/composer
!sudo chmod +x /usr/local/bin/composer
!composer --version

In [ ]:
# Clone repository
!git clone https://github.com/Herutriana44/sadoku
%cd sadoku

In [ ]:
# Install PHP dependencies
!composer install --no-interaction --prefer-dist 2>&1 | tail -20

In [ ]:
# Install Node dependencies
!npm install --legacy-peer-deps 2>&1 | tail -10

In [ ]:
# Install MySQL
!sudo apt-get install -y mysql-server -qq
!sudo service mysql start
!sudo service mysql status
print('MySQL started.')

In [ ]:
# Setup MySQL database and user
mysql_password = 'colab123'
!sudo mysql -e "CREATE DATABASE IF NOT EXISTS sadoku_db;"
!sudo mysql -e "CREATE USER IF NOT EXISTS 'colab_user'@'localhost' IDENTIFIED BY '{mysql_password}';"
!sudo mysql -e "GRANT ALL PRIVILEGES ON sadoku_db.* TO 'colab_user'@'localhost';"
!sudo mysql -e "FLUSH PRIVILEGES;"
print('Database ready.')

In [ ]:
# Write base .env (APP_URL will be set later after ngrok starts)
env_content = """APP_NAME="SADOKU"
APP_ENV=production
APP_KEY=
APP_DEBUG=false
APP_URL=http://localhost

LOG_CHANNEL=stack
LOG_LEVEL=error

DB_CONNECTION=mysql
DB_HOST=127.0.0.1
DB_PORT=3306
DB_DATABASE=sadoku_db
DB_USERNAME=colab_user
DB_PASSWORD=colab123

BROADCAST_DRIVER=log
CACHE_DRIVER=file
FILESYSTEM_DISK=local
QUEUE_CONNECTION=sync
SESSION_DRIVER=file
SESSION_LIFETIME=120

MAIL_MAILER=log
"""
with open('.env', 'w') as f:
    f.write(env_content)
print('.env written.')

In [ ]:
# Generate app key and run migrations
!php artisan key:generate --force
!php artisan migrate --force
!php artisan db:seed --force

# Running

In [ ]:
# Fix vite.config.js — must include @tailwindcss/vite plugin for Tailwind v4
vite_config = """import { defineConfig } from 'vite';
import laravel from 'laravel-vite-plugin';
import tailwindcss from '@tailwindcss/vite';

export default defineConfig({
    plugins: [
        tailwindcss(),
        laravel({
            input: [
                'resources/css/app.css',
                'resources/js/app.js',
            ],
            refresh: false,
        }),
    ],
    build: {
        // Emit manifest so Laravel @vite directive can resolve asset paths
        manifest: true,
        rollupOptions: {
            output: {
                // Predictable filenames prevent cache-busting issues in Colab
                entryFileNames: 'assets/[name].js',
                chunkFileNames: 'assets/[name].js',
                assetFileNames: 'assets/[name][extname]',
            },
        },
    },
});
"""
with open('vite.config.js', 'w') as f:
    f.write(vite_config)
print('vite.config.js updated.')

In [ ]:
# Fix AppServiceProvider to force HTTPS and correct root URL when behind ngrok proxy
provider = r"""<?php
namespace App\Providers;

use Illuminate\Support\ServiceProvider;
use Illuminate\Support\Facades\URL;

class AppServiceProvider extends ServiceProvider
{
    public function register(): void {}

    public function boot(): void
    {
        // When behind ngrok (or any reverse proxy), force the correct scheme and root URL
        // so asset() and route() helpers generate correct absolute URLs.
        $appUrl = config('app.url');
        if ($appUrl && str_starts_with($appUrl, 'https://')) {
            URL::forceScheme('https');
        }
        if ($appUrl && $appUrl !== 'http://localhost') {
            URL::forceRootUrl($appUrl);
        }
    }
}
"""
with open('app/Providers/AppServiceProvider.php', 'w') as f:
    f.write(provider)
print('AppServiceProvider.php updated.')

In [ ]:
import os, time, subprocess
from pyngrok import ngrok

# ── 1. Install pyngrok if needed ─────────────────────────────────────────────
!pip install pyngrok -q

# ── 2. Start ngrok FIRST so we know the public URL before building ────────────
ngrok.kill()  # kill any existing tunnels
ngrok.set_auth_token(AUTH_TOKEN_NGROK)

# Start PHP server in background
subprocess.Popen(
    ['php', 'artisan', 'serve', '--host=0.0.0.0', '--port=8000'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(2)  # wait for PHP server to be ready

tunnel = ngrok.connect(8000)
public_url = tunnel.public_url
print(f'ngrok URL: {public_url}')

# ── 3. Update .env with the real ngrok URL ────────────────────────────────────
with open('.env', 'r') as f:
    env_lines = f.readlines()

def set_env(lines, key, value):
    updated = [l for l in lines if not l.startswith(f'{key}=')]
    updated.append(f'{key}={value}\n')
    return updated

env_lines = set_env(env_lines, 'APP_URL', public_url)
env_lines = set_env(env_lines, 'ASSET_URL', public_url)
env_lines = set_env(env_lines, 'APP_ENV', 'production')
env_lines = set_env(env_lines, 'APP_DEBUG', 'false')

with open('.env', 'w') as f:
    f.writelines(env_lines)

print('.env updated with ngrok URL.')

# ── 4. Remove Vite HMR file (causes browser to look for localhost:5173) ───────
if os.path.exists('public/hot'):
    os.remove('public/hot')

# ── 5. Clear ALL caches before building ──────────────────────────────────────
!php artisan config:clear
!php artisan view:clear
!php artisan route:clear
!php artisan cache:clear

# ── 6. Build frontend assets (uses APP_URL from .env via laravel-vite-plugin) ─
print('Building assets...')
!npm run build

# ── 7. Cache config after build so Laravel reads it fast ─────────────────────
!php artisan config:cache
!php artisan route:cache
!php artisan view:cache

# ── 8. Fix storage permissions ────────────────────────────────────────────────
!chmod -R 775 storage bootstrap/cache

print(f'\n✅ App is live at: {public_url}')

# LINK Yang Di Klik

In [ ]:
print(f'\n✅ URL APLIKASI: {public_url}')

In [ ]:
# View logs if something breaks
!cat storage/logs/laravel.log 2>/dev/null | tail -50 || echo 'No log file yet.'

In [ ]:
# Run this cell to pull latest code and rebuild without restarting everything
!git pull
!npm run build
!php artisan config:clear
!php artisan view:clear
!php artisan config:cache
print(f'Rebuilt. URL: {public_url}')